In [1]:
import pandas as pd
import psycopg2

### Database connection

In [2]:
conn = psycopg2.connect(
    host="localhost",
    database="kenexaihackathon",
    user="postgres",
    password="09092002",
    port="5432"
)

### Load silver table

In [3]:
query = "SELECT * FROM silver.alerts"
df = pd.read_sql(query, conn)

print("Data Loaded:", df.shape)

Data Loaded: (1348, 11)


C:\Users\Palak\AppData\Local\Temp\ipykernel_15020\3100325966.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


### 1. Handle Missing Values

In [4]:
df["device_name"].fillna("Unknown_Device", inplace=True)
df["severity"].fillna("Info", inplace=True)

C:\Users\Palak\AppData\Local\Temp\ipykernel_15020\731420231.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["device_name"].fillna("Unknown_Device", inplace=True)
C:\Users\Palak\AppData\Local\Temp\ipykernel_15020\731420231.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

F

### 2. Standardize Severity

In [5]:
severity_map = {
    "warning": "Warning",
    "critical": "Critical",
    "failed": "Critical",
    "normal": "Info",
    "info": "Info"
}

df["severity"] = df["severity"].str.lower().map(severity_map).fillna("Unknown")

### 3. Remove Duplicate Alerts

In [6]:
df.drop_duplicates(subset="alert_id", inplace=True)

### 4. Normalize Alert Categories

In [7]:
def categorize_alert(alert):

    if pd.isna(alert):
        return "Other"

    alert = alert.lower()

    if "offline" in alert:
        return "Device Down"

    elif "disk" in alert:
        return "Disk Issue"

    elif "vpn" in alert:
        return "Network Issue"

    elif "cpu" in alert:
        return "CPU Issue"

    return "Other"

In [8]:
df["alert_category"] = df["alert_type"].apply(categorize_alert)

### 5. Timestamp Processing

In [9]:
df["occurred_at"] = pd.to_datetime(df["occurred_at"])

df["alert_hour"] = df["occurred_at"].dt.hour
df["alert_day"] = df["occurred_at"].dt.date

### 6. Severity Score

In [10]:
severity_score = {
    "Info": 1,
    "Warning": 2,
    "Critical": 3
}

df["severity_score"] = df["severity"].map(severity_score)

### 7. Device Alert Count

In [11]:
device_counts = df.groupby("device_name").size()

df["device_alert_count"] = df["device_name"].map(device_counts)

print("Preprocessing Completed")

Preprocessing Completed


In [12]:
df.head()

,alert_id,source_system,device_name,device_identifier,organization_name,network_name,alert_type,severity,alert_message,occurred_at,ingestion_time,alert_category,alert_hour,alert_day,severity_score,device_alert_count
0,MTA4MzgxOTEyOTIzODk1MDY1MywxMzA0MzQ1NzYxNzYxND...,Auvik,DMG-SLO-AP02,MTA4MzgxOTEyOTIzODk1MDY1MywxMDkzNzg0NDM2MzE1Nj...,Ambient Enterprises DMG San Luis Obispo,None,Access Point Offline,Warning,This network element has gone offline Ubiqui...,2025-11-25 14:44:06.117,2026-03-14 14:28:23.482325,Device Down,14,2025-11-25,2.0,19
1,MTE2NzQyMDIwMzUzMTg2Njg3NywxMzA3NjY4MjcxMzU1OD...,Auvik,GigabitEthernet1/0/3 on dyoOBTextsw01.dyopath.com,MTE2NzQyMDIwMzUzMTg2Njg3NywxMTg1MTI3NjkwNTA2OT...,DYOPATH Oakbrook Terrace,None,Interface Status Mismatch,Warning,Interface GigabitEthernet1/0/3 (Connection to ...,2025-11-25 14:52:29.787,2026-03-14 14:28:23.482325,Other,14,2025-11-25,2.0,2
2,MTA4MzgxOTEyOTIzODk1MDY1MywxMzA0MzQ1NzYxNzYyNj...,Auvik,DMG-SLO-AP02,MTA4MzgxOTEyOTIzODk1MDY1MywxMDkzNzg0NDM2MzE1Nj...,Ambient Enterprises DMG San Luis Obispo,None,Access Point Offline,Warning,Device Online Status is now equal to Online,2025-11-25 14:54:05.596,2026-03-14 14:28:23.482325,Device Down,14,2025-11-25,2.0,19
3,MTA4MzgxOTEyOTIzODk1MDY1MywxMzA0MzQ1NzYxNzYyNj...,Auvik,DMG-SLO-AP01,MTA4MzgxOTEyOTIzODk1MDY1MywxMDkzNzg0NDM2MjUzNT...,Ambient Enterprises DMG San Luis Obispo,None,Access Point Offline,Warning,This network element has gone offline Ubiqui...,2025-11-25 14:54:05.596,2026-03-14 14:28:23.482325,Device Down,14,2025-11-25,2.0,30
4,MTEzOTQ0OTI3MTk4NzI3ODU4OSwxMzA0MzQ1NzY5NzEzND...,Auvik,asa5515x-01.kslinc.net,MTEzOTQ0OTI3MTk4NzI3ODU4OSwxMzA0MzQ1NzcxMzY2Nz...,HVP SLHV Data Center,None,VPN remote gateway is Lost,Critical,The VPN remote gateway 12.207.114.41 has been ...,2025-11-25 14:54:46.402,2026-03-14 14:28:23.482325,Network Issue,14,2025-11-25,3.0,60


In [13]:
from sqlalchemy import create_engine

engine = create_engine("postgresql+psycopg2://", creator=lambda: conn)

df.to_sql(
    "alerts_clean",
    con=engine,
    schema="silver",
    if_exists="replace",
    index=False,
    method="multi"
)

print("Loaded to silver.alerts_clean")

Loaded to silver.alerts_clean


In [14]:
df.to_csv("../../data/processed/alerts_clean.csv", index=False)